In [204]:
import xarray as xr
import numpy as np

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*Dataset.dims.*"
)

from tensor_calculus import Tensor

%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [205]:
D0 = Tensor(xr.DataArray(np.zeros(2), dims=['i']), label='u_i')
D1 = Tensor(xr.DataArray(np.zeros([2,2]), dims=['i', 'j']), label='@_ju_i')
D2 = Tensor(xr.DataArray(np.zeros([2,2,2]), dims=['i', 'j','k']), label='@_k@_ju_i')
D3 = Tensor(xr.DataArray(np.zeros([2,2,2,2]), dims=['i', 'j','k','m']), label='@_m@_k@_ju_i')
D4 = Tensor(xr.DataArray(np.zeros([2,2,2,2,2]), dims=['i', 'j','k','m','n']), label='@_n@_m@_k@_ju_i')

In [206]:
D2.set_symmetric_indices(['j','k'])
D3.set_symmetric_indices(['j','k', 'm'])
D4.set_symmetric_indices(['j','k', 'm', 'n'])

# Smagorinsky model

In [9]:
((S1*S1).contract(['i','k']).contract(['j','m']).sqrt()*S2).contract(['j', 'k'])

In [10]:
(S1*S1).contract(['i','k']).contract(['j','m']).hash_array

<xarray.DataArray ()> Size: 8B
array(1.79999454)

In [11]:
(S1*D1).contract(['i','m']).contract(['j','k']).hash_array

<xarray.DataArray ()> Size: 8B
array(1.79999454)

In [12]:
(D1-S1-O1).hash_array

<xarray.DataArray (i: 2, j: 2)> Size: 32B
array([[0., 0.],
       [0., 0.]])
Dimensions without coordinates: i, j

In [191]:
from IPython.display import Math, display
for t in (D4).contract_to_rank_one(add_perp=True):
    display(Math(f"{t.rename()._repr_latex_()}"))
    print(t.hash_array.values)

<IPython.core.display.Math object>

[0.1000744  0.55893368]


<IPython.core.display.Math object>

[-0.71734222  0.11478089]


<IPython.core.display.Math object>

[0.21485529 1.27627591]


In [146]:
from IPython.display import Math, display
for t in (D4).contract_to_rank_one(add_perp=True):
    display(Math(f"{t.rename()._repr_latex_()}"))
    print(t.hash_array.values)

<IPython.core.display.Math object>

[-1.64230778  0.39468858]


<IPython.core.display.Math object>

[-0.45589268 -0.81541045]


<IPython.core.display.Math object>

[-2.45771823  0.85058127]


In [151]:
from IPython.display import Math, display
for t in (D2*D1).contract_to_rank_one(add_perp=True):
    display(Math(f"{t.rename()._repr_latex_()}"))
    #print(t.hash_array.values)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [194]:
from IPython.display import Math, display
for t in (D3*D0).contract_to_rank_one():
    display(Math(f"{t.rename()._repr_latex_()}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [195]:
from IPython.display import Math, display
for t in (D3*D0*D1).contract_to_rank_one():
    display(Math(f"{t.rename()._repr_latex_()}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Test library construction

In [207]:
from IPython.display import Math, display
from itertools import combinations_with_replacement

def print_basis(results):
    print('Library of tensors has length', len(results))
    for t in results:
        display(Math(f"{t.rename()._repr_latex_()}"))


def construct_basis_of_tensors(initial_tensors, their_derivatives=None, max_nonlinearity = 1, max_derivative=6, add_perp=False, add_advection=False):
    results = []

    for nonlinearity in range(1, max_nonlinearity+1):
        for idx in combinations_with_replacement(range(0, len(initial_tensors)), nonlinearity):
            tensor = initial_tensors[idx[0]]
            for i in range(1, len(idx)):
                tensor = tensor*initial_tensors[idx[i]]

            sum_derivative = sum([their_derivatives[idx] for idx in idx])
            if sum_derivative > max_derivative:
                continue
            #display(Math(tensor._repr_latex_()+ f'${sum_derivative}$'))

            results.extend(tensor.contract_to_rank_one(add_perp=add_perp))

    final_results = results

    if add_advection:
        for t in results:
            for tt in results:
                if t.label.count('@') + tt.label.count('@') < max_derivative:
                    final_results.append((t * tt.diff(param, grid)).contract(['i', 'j']).rename())


    return final_results

In [208]:
basis = construct_basis_of_tensors([D0, D1, D2], 
                                    [0, 1, 2],
                                    max_nonlinearity=2)
print_basis(basis)

Library of tensors has length 15


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [209]:
basis = construct_basis_of_tensors([D0, D1, D2], 
                                    [0, 1, 2],
                                    max_nonlinearity=2, add_perp=True)
print_basis(basis)

Library of tensors has length 41


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [214]:
basis[21]

In [216]:
basis[24]

In [213]:
basis[24].hash_array

<xarray.DataArray (i: 2)> Size: 16B
array([-2.21862055, -0.38512226])
Dimensions without coordinates: i

In [217]:
basis[21].hash_array

<xarray.DataArray (i: 2)> Size: 16B
array([-1.66217863, -0.83512186])
Dimensions without coordinates: i

In [223]:
(D1*D2).contract(['j','k']).contract(['i','n'])

In [224]:
(D1*D2).perp('j').contract(['j','k']).perp('i').contract(['i','n'])